> 📓 **Lesson 1.9 — Part 1 of 4: Time Series — Making Time a First-Class Column**
>
> This notebook was split out of the original single `eda_advanced.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.9.
>
> Other notebooks in this set: `Part_2_data_integration.ipynb`, `Part_3_aggregation_reporting.ipynb`, `Part_4_table_to_decision.ipynb`

# Lesson 1.9: EDA Advanced — Data Wrangling & Analysis

Lesson 1.8 asked *"can I trust this data?"*. This lesson asks the next question:
**what is the pattern, and what should we do about it?**

Clean rows on their own answer nothing. You have to put time on the index, join in the tables that
give the rows meaning, reshape them, and group them. That is the whole job here.

**Structure — the four learning outcomes, in order:**
* **Part 1: Time Series** — *parse* dates, then resample and roll them.
* **Part 2: Data Integration** — *merge* tables, and convert wide ↔ long.
* **Part 3: Aggregation & Reporting** — *aggregate* with `groupby`, `pivot_table`, `crosstab`.
* **Part 4: From Table to Decision** — *apply* all of it to answer the owner's actual question.

**How to read the code cells:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 180 minutes.** One business problem, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Time Series | **Parse** dates; `resample`, `rolling`, `shift` | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Integration | **Merge** tables; `melt` / `pivot` (wide ↔ long) | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Aggregation & Reporting | **Aggregate**: `groupby`, `pivot_table`, `crosstab` | 45 min |
> | **Part 4** | From Table to Decision | **Apply** split-apply-combine to the real question | 20 min |
>
> **The spine:** one business problem — *The Daily Grind*, a four-outlet café chain — and one main
> file, `data/daily_sales.csv`, from start to finish. Small hand-built tables appear alongside as
> *drills*: they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`.


### The four beats of every summary

Lesson 1.8 gave you four beats for every fix: **find it → decide → apply → verify.**
Summarising has its own four, and every table we build today follows them:

| Beat | Ask yourself | |
|---|---|---|
| **1. Question** | What decision does this number serve? | *Renew the Marina Bay lease — yes or no?* |
| **2. Grain** | One row per **what**? | *One row per outlet, per month* |
| **3. Aggregation** | Sum, mean or count — and **why that one**? | *Sum for totals, mean for efficiency* |
| **4. Check** | Does the total still tie back? | *Grouped total == ungrouped total* |

Beat 4 is the one everyone skips. In Part 2 it catches a join that silently deletes $61,310.


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Two toolkits. `pd` and `np` are just short nicknames, so we can type `pd.something`
#    instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np

# 👉 Housekeeping only. Pandas renames a few option strings between versions and shouts
#    about it; this keeps those notices out of our output. Nothing to learn here.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# 👉 The spine. One row per outlet, per day, per part of the day (Morning/Midday/Evening).
#    18 months of trading for a four-outlet café chain, plus a pop-up kiosk.
#    `parse_dates=["date"]` tells pandas: this column is not text, it is a date. More on that in 1.1.
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])

sales.head()


In [ ]:
# 👉 The 1.8 habit still applies: look before you leap. Shape, types, holes.
print("rows, columns:", sales.shape)
sales.info()


### 🎬 Why this matters — the flat line that hides everything

**The situation.** *The Daily Grind* runs four cafés in Singapore. Revenue has been flat for two
quarters. The Marina Bay lease is up for renewal this month, rent is $9,600, and the owner has to
sign or walk away. She sends you the sales export and asks one question: **what is going on?**

Run the next two cells. The first is the number she already has. The second is the same number,
split by outlet.

> Do not worry about how these two lines work yet — that is Part 1 and Part 3. Just read the output.


In [ ]:
# 👉 Total revenue per quarter for the whole chain -- the headline the owner already has.
#    (`.to_period("Q")` labels each date with its calendar quarter.)
chain_by_quarter = sales.groupby(sales["date"].dt.to_period("Q"))["revenue_sgd"].sum().round(0)

chain_by_quarter


In [ ]:
# 👉 The same revenue, but one column per outlet. Same data. Same period. Different question.
by_outlet = sales.pivot_table(
    index=sales["date"].dt.to_period("Q"),   # down the side: quarter
    columns="outlet_id",                     # across the top: outlet
    values="revenue_sgd",                    # the number in the middle
    aggfunc="sum",                           # how to squash it: add it up
).round(0)

by_outlet


**Read the second table.** OUT-03 falls from about \$138k a quarter to about \$100k. OUT-04 climbs
from about \$96k to about \$131k. One is dying, one is growing, and they move by almost the same
amount — so the chain total barely twitches.

The flat line was never the story. It was **two opposite stories cancelling out**.

No amount of cleaning would have found this. Cleaning gives you rows you can trust; only grouping
turns them into an answer. Three more things you cannot see yet, and will by the end of the session:

1. OUT-03's fall is not a slope. It is a **step**, on one specific week. (Part 1)
2. There is a fifth outlet in this file that does not exist in the outlet list. (Part 2)
3. OUT-03 is still staffed for the revenue it used to make. (Part 3)

Write those three down. We will tick them off.


---

## Part 1: Time Series — Making Time a First-Class Column

**Learning outcome 1:** *Parse and manipulate datetime data to perform time-based resampling and
rolling window calculations.*

**Goal:** the owner's question is about *change over time*. Before pandas can answer anything about
time, it has to know that a date is a date — not a piece of text that happens to look like one.

⏱️ ~45 min including Group Exercise 1


### 1.1: A date is a type, not a string

This is the whole idea of the section, so it is worth ten seconds of proof.


In [ ]:
# 👉 Load the same file again, but WITHOUT telling pandas that `date` is a date.
#    `object` in the output below means "text". Pandas has no idea these are dates.
raw = pd.read_csv("../data/daily_sales.csv")

raw["date"].dtype


In [ ]:
# 👉 Here is why that matters. As text, "2024-10-02" sorts before "2024-9-30" -- alphabetically,
#    "1" comes before "9". Dates as text sort like words, not like time.
text_dates = pd.Series(["2024-10-02", "2024-9-30", "2024-11-01"])

text_dates.sort_values()


In [ ]:
# 👉 `pd.to_datetime` converts text into real timestamps. Now sorting means what you expect.
real_dates = pd.to_datetime(text_dates)

real_dates.sort_values()


In [ ]:
# 👉 Not every system writes dates the American way. "01/06/2025" is 1 June in Singapore and
#    6 January in the US -- and pandas cannot know which you meant. So tell it.
#    `format=` spells the layout out: %d day, %m month, %Y four-digit year.
uk_style = pd.Series(["01/06/2025", "02/06/2025", "03/06/2025"])

pd.to_datetime(uk_style, format="%d/%m/%Y")


In [ ]:
# 👉 `dayfirst=True` is the shorter way to say the same thing when the layout is consistent.
pd.to_datetime(uk_style, dayfirst=True)


> **This is the single most common silent bug in date handling.** Without `format=` or
> `dayfirst=True`, pandas guesses from the first value it can parse and then applies that guess to
> the whole column. Days 1–12 of a month parse "successfully" under the wrong reading, so
> `"03/06/2025"` becomes 6 March and no error is raised. Your report is then wrong by three months
> and looks fine.
>
> If your dates genuinely change format row to row, pandas 2.0+ has `format="mixed"`. In the `pds`
> environment (pandas 1.5) you would clean the column first — one format at a time.


In [ ]:
# 👉 Once a column is a real date, the `.dt` accessor unlocks date questions --
#    exactly like `.str` unlocked text questions in 1.8.
print("first day:", sales["date"].min().date())
print("last day: ", sales["date"].max().date())
print("days covered:", (sales["date"].max() - sales["date"].min()).days + 1)

sales["date"].dt.day_name().sample(5)


In [ ]:
# 👉 Pull date parts out into their own columns so we can group by them later.
sales["month"] = sales["date"].dt.to_period("M")   # 2024-01, 2024-02, ...
sales["weekday"] = sales["date"].dt.day_name()     # Monday, Tuesday, ...
sales["is_weekend"] = sales["date"].dt.dayofweek >= 5   # Saturday=5, Sunday=6

sales[["date", "month", "weekday", "is_weekend"]].sample(5)


> **Beat 2 in action.** `sales` has one row per outlet **per day per daypart** — three rows per
> outlet per day. Any total you take without saying which grain you want will quietly mix them.


### 1.2: The DatetimeIndex — put time on the index

A `DatetimeIndex` is a date column promoted to be the row label. It is what unlocks `.resample()`,
`.rolling()`, and slicing by `"2025-06"` instead of writing a filter.


In [ ]:
# 👉 One number per day for the whole chain: add up every outlet and every daypart on that date.
#    The result is a Series whose INDEX is the date -- that is a DatetimeIndex.
chain_daily = sales.groupby("date")["revenue_sgd"].sum()

print(type(chain_daily.index).__name__)
chain_daily.head()


In [ ]:
# 👉 With dates on the index you can slice with plain strings. This is one month:
chain_daily["2025-06"].head()


In [ ]:
# 👉 ...and this is a range. Both ends are INCLUDED with `.loc` on dates (unlike normal Python).
chain_daily.loc["2024-11-01":"2024-11-05"]


In [ ]:
# 👉 Drill on a hand-built series, so the mechanics are visible.
#    `date_range` makes a run of dates; here: 6 days starting 1 Jan.
idx = pd.date_range("2024-01-01", periods=6, freq="D")
drill = pd.Series([10, 12, 9, 15, 11, 14], index=idx)

drill


In [ ]:
# 👉 Real data has gaps. Drop two days, then `reindex` back onto the full calendar:
#    the missing days come back as NaN instead of silently disappearing.
gappy = drill.drop([pd.Timestamp("2024-01-03"), pd.Timestamp("2024-01-04")])

gappy.reindex(idx)


> **Why the gap matters.** A missing day and a zero-revenue day look identical in a chart, and mean
> opposite things: "closed for a public holiday" vs "open and sold nothing". `reindex` makes the
> difference visible before you average anything.


### 1.3: `resample` — change the grain of time

`resample` is `groupby` for dates. You give it a frequency; it buckets the rows and aggregates
each bucket. **Beat 3 applies:** the frequency is the grain, the aggregation is your choice, and
choosing wrong gives a confident wrong answer.

| Alias | Bucket |
|---|---|
| `D` | calendar day |
| `W` | week (ending Sunday by default) |
| `M` | month end |
| `Q` | quarter end |
| `A` / `Y` | year end |


In [ ]:
# 👉 Daily -> monthly, adding up each month. This is the owner's headline number.
monthly_chain = chain_daily.resample("M").sum().round(0)

monthly_chain.tail(8)


In [ ]:
# 👉 The same resample with `.mean()` answers a DIFFERENT question: an average TRADING DAY.
#    Sum is distorted by month length (February is short); mean is not. Neither is "correct" --
#    they answer different questions. That is beat 3.
compare = pd.DataFrame({
    "total_revenue": chain_daily.resample("M").sum().round(0),
    "avg_day": chain_daily.resample("M").mean().round(0),
    "trading_days": chain_daily.resample("M").size(),
})

compare.tail(6)


> **Look at February 2025.** The total drops and the average day barely moves. The "drop" was 28
> days versus 31, plus Chinese New Year — not a business problem. A manager shown only the totals
> column would go looking for a cause that does not exist.


In [ ]:
# 👉 Resampling upwards (finer) instead of downwards creates rows that did not exist, so you must
#    say how to fill them. `.asfreq()` leaves NaN; `.ffill()` carries the last value forward.
weekly = chain_daily.resample("W").sum()

weekly.resample("D").asfreq().head(4)


### 1.4: `rolling` — smooth the noise to see the shape

Daily café revenue swings wildly between weekdays and weekends. A rolling (moving) average replaces
each day with the average of it and the days before it, which strips out the weekly rhythm and
leaves the trend.


In [ ]:
# 👉 Marina Bay only. One number per day for OUT-03.
marina = sales[sales["outlet_id"] == "OUT-03"].groupby("date")["revenue_sgd"].sum()

marina.loc["2024-10-28":"2024-11-03"].round(0)


In [ ]:
# 👉 A 7-day rolling mean: each value is the average of that day and the 6 before it.
#    7 days = exactly one week, so it cancels the weekday/weekend cycle.
#    The first 6 values are NaN -- there is nothing behind them to average.
marina_7d = marina.rolling(window=7).mean()

marina_7d.head(9).round(0)


In [ ]:
# 👉 Now put the raw and the smoothed side by side across the first week of November 2024.
#    Ignore the daily zig-zag and read the `smooth_7d` column downwards.
pd.DataFrame({
    "raw": marina.round(0),
    "smooth_7d": marina_7d.round(0),
}).loc["2024-10-28":"2024-11-14"]


In [ ]:
# 👉 A 28-day window smooths harder. Compare the monthly averages either side of 4 Nov 2024:
before = marina.loc["2024-09-01":"2024-11-03"].mean()
after = marina.loc["2024-11-04":"2025-01-31"].mean()

print(f"average day before 4 Nov 2024: ${before:,.0f}")
print(f"average day after  4 Nov 2024: ${after:,.0f}")
print(f"change: {(after / before - 1) * 100:,.1f}%")


> **Tick off finding #1.** That is not a slope, it is a **step** — one week, then a new normal.
> Slopes and steps have different causes. A slope says "we are slowly losing our regulars"; a step
> says "something happened on that date". (A competitor opened next door on 4 November 2024.)
>
> A rolling average is the cheapest tool you own for telling those two apart.


In [ ]:
# 👉 `.shift()` moves the values down by n rows, which lets you compare a period with the one
#    before it. `.pct_change()` is the same idea, packaged: (this - previous) / previous.
mom = pd.DataFrame({
    "revenue": monthly_chain,
    "prev_month": monthly_chain.shift(1),
    "change_pct": (monthly_chain.pct_change() * 100).round(1),
})

mom.tail(6)


> **Do not trust a percentage until you know what is in the denominator.** March 2025 is up 25%
> and June is down 13% — and neither is about the cafés. A pop-up kiosk traded from March to May
> and is in this total. Month-on-month change is the most over-quoted number in business reporting
> precisely because it moves for reasons like that.


### 1.5: The payoff — resample per outlet

One `resample` on the chain hid the story. The same resample, done per outlet, reveals it.


In [ ]:
# 👉 Reading it inside out:
#    (1) pivot so each outlet is a column and each date is a row,
#    (2) resample those rows to month-end totals.
#    `pivot_table` with `aggfunc="sum"` collapses the three dayparts per day into one number.
by_outlet_daily = sales.pivot_table(
    index="date", columns="outlet_id", values="revenue_sgd", aggfunc="sum"
)

monthly_by_outlet = by_outlet_daily.resample("M").sum()

monthly_by_outlet.tail(6).round(0)


In [ ]:
# 👉 Beat 4: the check. Do the per-outlet monthly totals still add up to the chain total?
#    If this prints False, we lost rows somewhere and every number above is suspect.
#    `np.isclose` compares with a tiny tolerance -- decimals of a cent should not fail a check.
grouped_total = monthly_by_outlet.sum().sum()
raw_total = sales["revenue_sgd"].sum()

print("per-outlet total ties back to the raw total:", np.isclose(grouped_total, raw_total))
print(f"grouped: ${grouped_total:,.2f}")
print(f"raw:     ${raw_total:,.2f}")


> **Compare exactly the thing you care about.** Note that we checked the *unrounded* totals. Had we
> rounded each monthly figure to the nearest dollar first and then added them up, the check would
> have failed by a few dollars — 75 separate roundings — and sent us hunting for a data-loss bug that
> did not exist. Round for display, never before a comparison.


### 🛠️ Group Exercise 1 — Time (8 min)

Resample Holland Village (`OUT-04`) to monthly totals and find its best month.

*Hint:* collapse the three dayparts into one number per day first (`groupby("date")`), then resample that daily series to month ends (`resample("M")`).

---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Holland Village (`OUT-04`) monthly totals, and its best month.**

In [ ]:
hv_daily = sales[sales["outlet_id"] == "OUT-04"].groupby("date")["revenue_sgd"].sum()
hv_monthly = hv_daily.resample("M").sum()

print("best month:", hv_monthly.idxmax().date(), "->", f"${hv_monthly.max():,.0f}")
# hv_monthly.round(0).tail(8)

March 2025 is the peak (~\$45,970), and the whole column climbs from ~\$31k in January 2024. `.idxmax()` gives the *label* of the largest value (the month-end date); `.max()` gives the value itself. Note the index labels are month **ends** — `resample("M")` stamps each bucket with its last day.

---

# ☕ Break — 10 minutes

**Where we are:** time is now a real column, and you can change its grain (`resample`) and smooth
it (`rolling`). You found the step at Marina Bay.

**Next up:** Part 2 — the sales file only knows outlet *codes*. To say anything about rent, region
or staffing, we have to join in the other tables.


📂 **Open** `Part_2_data_integration.ipynb` to continue.